In [ ]:
%%html

<!-- UI_RUNNER: Data Structures for Courses --> 

<script>

// =======================================
// COURSE OBJECTS - Page-level course data
// =======================================
class Course {
  constructor(code, title, subtitle, mission, unlockThreshold, learningJourney = [], zoneTitle = '') {
    this.code = code;
    this.title = title;
    this.subtitle = subtitle;
    this.mission = mission;
    this.unlockThreshold = unlockThreshold;
    this.learningJourney = learningJourney;
    this.zoneTitle = zoneTitle || `${code} Zone - ${title}`;
  }

  isUnlocked(totalClicks) {
    return totalClicks >= this.unlockThreshold;
  }

  getRemainingClicks(totalClicks) {
    return Math.max(0, this.unlockThreshold - totalClicks);
  }
}

// Define courses as page-level objects
window.ocsCourses = {
  CSSE: new Course(
    'CSSE',
    'CSSE - Software Engineering',
    'Games, Foundation',
    'Computer Science and Software Engineering 1,2 prepares students for the AP Computer Science pathway. This course focuses on teaching the JavaScript programming language, object-oriented programming, and developing algorithmic thinking skills through game development projects.',
    5,
    ['Game Builder', 'JS Fundamentals (Code Runner)', 'GitHub Project and VSCode', 'Game Objects', 'Sprite Animation Studio', 'WASD Maze Navigator'],
    'CSSE Zone - Software Engineering Foundation'
  ),
  CSP: new Course(
    'CSP',
    'CSP - Computer Science Principles',
    'Projects, Principles',
    'AP Computer Science Principles (CSP 1, CSP 2, DS 1) incorporates College Board curriculum through Project-base learning.  Teams, Data, Coding, Computer Systems, and Networks are utilized to build community serving projects and engage client in discussions about solutions.',
    10,
    ['Python Fundamentals', 'Algorithm Design & Analysis', 'Data Structures Exploration', 'Flask Web Applications', 'SQL Databases', 'Team Project Development'],
    'CSP Zone - Principles of Programming'
  ),
  CSA: new Course(
    'CSA',
    'CSA - Computer Science A',
    'Code, Architecture',
    'AP Computer Science A (CSA 1, CSA 2, DS 2) is an in-depth course focusing on programming, algorithms, and data structures. Students prepare for AP exam by architecting and building systems, focusing on code quality, becoming an algorithmic problem solver, establishing fluency in JavaScript, Java and OOP.',
    15,
    ['Java OOP Fundamentals', 'Classes and Objects', 'Inheritance & Polymorphism', 'Arrays & ArrayLists', 'Recursion Patterns', 'Spring Framework Projects'],
    'CSA Zone - Computer Science A'
  ),
  CSH: new Course(
    'CSH',
    'CSH - Computer Science Honors',
    'Research, Capstone',
    'Computer Science "H" is a 2-trimester, senior-only interdisciplinary honors course serving as the Pathway Capstone. Students complete a research, design, and coding thesis culminating in a presentation to sponsoring professionals.',
    20,
    ['Capstone Project Planning', 'Research & Problem Definition', 'Stakeholder Engagement', 'System Design & Prototyping', 'Technical Documentation', 'Public Presentation'],
    'CSH Zone - Computer Science Honors'
  )
};

// Expose classes globally for use in other cells
window.Course = Course;

</script>


In [ ]:
%%js

// GAME_RUNNER: Journey launcher | hide_edit: true, panel: home_zone_journey, slot: right, layout: row, ratio: 35-65, gap: 1rem, width: 100%, height: 320px

import GameControl from '@assets/js/GameEnginev1.1/essentials/GameControl.js';
import Clicker from '@assets/js/GameEnginev1.1/essentials/Clicker.js';

/**
┌─────────────────────────┐
│   ZoneJourneyGame       │ ← Configured to spawn 3 cookies (spawn.count), 
└───────────┬─────────────┘
            │ cookie-click event to share click state
            ↓
┌─────────────────────────┐
│   ZoneStateManager      │ ← Compares clicks to thresholds
└───────────┬─────────────┘
            │ manages zone states and local storage
            ↓
┌─────────────────────────┐
│  ZoneButtonRenderer     │ ← Shows/hides zone panels based on state
└─────────────────────────┘
*/

class ZoneJourneyGame {
  constructor(gameEnv) {
    const path = gameEnv.path;
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;

    // Cookie sprites to randomly assign
    const cookieSprites = [
      path + '/hacks/cookie-clicker/assets/baseCookie.png', 
      path + '/hacks/cookie-clicker/assets/grandma.png'
    ];

    const spriteData = {
      id: 'Journey Token',
      greeting: "Click cookies to unlock course zones!",
      src: cookieSprites[0],
      SCALE_FACTOR: 7,
      pixels: { height: 512, width: 512 },
      INIT_POSITION: { x: 0, y: 0 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1, wiggle: 0.15 },
      up: { row: 0, start: 0, columns: 1, wiggle: 0.15 },
      left: { row: 0, start: 0, columns: 1, wiggle: 0.15 },
      right: { row: 0, start: 0, columns: 1, wiggle: 0.15 },
      hitbox: { widthPercentage: 0.2, heightPercentage: 0.2 },
      walkingArea: { xMin: 0, xMax: width, yMin: 0, yMax: height },
      speed: 2,  // Constant speed - easy gameplay
      direction: { x: 1, y: 1 },
      interact: function(clicks, objectId) {
        // Dispatch cookie click event to increment global counter
        window.dispatchEvent(new CustomEvent('cookie-click'));
        
        const journey = window.ocsZoneJourney;
        const metric = document.querySelector('[id^="gamerunner-home-gamified-mvp"][id$="-metric"]');
        if (metric) {
          metric.textContent = "Total Clicks: " + journey.totalClicks;
        }
      }
    };

    this.classes = [{
      class: Clicker,
      data: spriteData,
      spawn: {
        count: 3,  // 3 cookies
        ranges: {
          INIT_POSITION: {
            x: [0, Math.max(0, width - 128)],
            y: [0, Math.max(0, height - 128)]
          }
        },
        pickOne: { 
          src: cookieSprites  // Randomly pick sprite for each cookie
        }
      }
    }];
  }
}

export const gameLevelClasses = [ZoneJourneyGame];
export { GameControl };

In [ ]:
%%html

<!-- UI_RUNNER: Helper classes and data structures for gamified progress tracking and course management --> 

<script>

/** ===============================================
 ZONE STATE MANAGER - Manages zone states 
 ==================================================
 ocsZoneProgress:
 {
    "totalClicks": 20,
    "unlocked": {
        "CSSE": true,
        "CSP": true,
        "CSA": true,
        "CSH": true
    },
    "selectedZone": "CSSE"
}
*/

class ZoneStateManager {
  constructor(storageKey = 'ocsZoneProgress') {
    this.storageKey = storageKey;
    this.totalClicks = 0;
    this.unlocked = {};
    this.selectedZone = null;
    this.load();
  }

  load() {
    const saved = localStorage.getItem(this.storageKey);
    if (saved) {
      try {
        const data = JSON.parse(saved);
        this.totalClicks = data.totalClicks || 0;
        this.unlocked = data.unlocked || {};
        this.selectedZone = data.selectedZone || null;
      } catch (e) {
        console.error('Error loading progress:', e);
      }
    }
  }

  save() {
    localStorage.setItem(this.storageKey, JSON.stringify({
      totalClicks: this.totalClicks,
      unlocked: this.unlocked,
      selectedZone: this.selectedZone
    }));
  }

  incrementClick() {
    this.totalClicks++;
    this.save();
  }

  clear() {
    localStorage.removeItem(this.storageKey);
    this.totalClicks = 0;
    this.unlocked = {};
    this.selectedZone = null;
  }

  selectZone(zoneCode) {
    this.selectedZone = zoneCode;
    this.save();
  }

  isUnlocked(zoneCode) {
    return this.unlocked[zoneCode] || false;
  }

  setUnlocked(zoneCode, unlocked) {
    this.unlocked[zoneCode] = unlocked;
  }
}


// Expose classes globally for use in other cells
window.ZoneStateManager = ZoneStateManager;

</script>


In [ ]:
%%html
<!-- UI_RUNNER: Zone unlock panel with buttons | panel: home_zone_journey, slot: left, layout: row, ratio: 35-65, gap: 1rem -->

<!-- Styles defined in: _sass/open-coding/forms/home-gamified.scss -->

<div id="zone-unlock-panel">
  <h3>Progress Tracker</h3>
  <p class="zone-subtitle">Cookies: <span id="total-clicks">0</span> <button id="clear-cookies-btn" style="margin-left: 10px; padding: 4px 8px; font-size: 0.85em;">Clear Cookies</button></p>
  <!-- Buttons will be dynamically generated here by ZoneButtonRenderer renderButtons method -->
  <div id="zone-buttons-container"></div>
</div>

<script>

// ============================================
// DOM BUTTON RENDERING - Updates UI
// ============================================
class ZoneButtonRenderer {
  constructor(stateManager, courses) {
    this.stateManager = stateManager;
    this.courses = courses;
    this.renderButtons();
  }

  renderButtons() {
    const container = document.getElementById('zone-buttons-container');
    if (!container) return;
    
    // Clear existing buttons
    container.innerHTML = '';
    
    // Create buttons for each course
    const courseKeys = Object.keys(this.courses);
    courseKeys.forEach(courseCode => {
      const course = this.courses[courseCode];
      
      const button = document.createElement('button');
      button.className = 'zone-btn locked';
      button.setAttribute('data-zone', courseCode);
      button.disabled = true;
     
      // Button template with title, subtitle, and lock status 
      button.innerHTML = `
        <div class="zone-btn-title">${courseCode}</div>
        <div class="zone-btn-subtitle">${course.subtitle}</div>
        <div class="zone-btn-lock">🔒 ${course.unlockThreshold} clicks</div>
      `;
      
      container.appendChild(button);
    });
  }

  updateAllButtons() {
    const courseKeys = Object.keys(this.courses);
    
    courseKeys.forEach(courseCode => {
      this.updateButton(courseCode);
    });
    
    this.updateZoneSectionVisibility();
    this.stateManager.save();
  }

  updateButton(courseCode) {
    const course = this.courses[courseCode];
    const totalClicks = this.stateManager.totalClicks;
    const unlocked = course.isUnlocked(totalClicks);
    const btn = document.querySelector('.zone-btn[data-zone="' + courseCode + '"]');
    
    if (!btn) return;
    
    // Update unlock state in state manager
    this.stateManager.setUnlocked(courseCode, unlocked);
    
    // Update button state
    if (unlocked) {
      btn.classList.remove('locked');
      btn.classList.add('unlocked');
      btn.disabled = false;
      const lockDiv = btn.querySelector('.zone-btn-lock');
      if (lockDiv) lockDiv.textContent = '✅';
    } else {
      btn.classList.remove('unlocked');
      btn.classList.add('locked');
      btn.disabled = true;
      const lockDiv = btn.querySelector('.zone-btn-lock');
      const remaining = course.getRemainingClicks(totalClicks);
      if (lockDiv) lockDiv.textContent = '🔒 ' + remaining + ' more';
    }
    
    // Update active state
    if (this.stateManager.selectedZone === courseCode) {
      btn.classList.add('active');
    } else {
      btn.classList.remove('active');
    }
  }

  updateZoneSectionVisibility() {
    const courseKeys = Object.keys(this.courses);
    
    courseKeys.forEach(courseCode => {
      const section = document.getElementById('zone-section-' + courseCode);
      const isActive = this.stateManager.selectedZone === courseCode && 
                      this.stateManager.isUnlocked(courseCode);
      
      if (section) {
        section.style.display = isActive ? 'block' : 'none';
      }
    });
  }

  // This displays clicks (game -> state manager -> renderer)
  updateClickDisplay() {
    const display = document.getElementById('total-clicks');
    if (display) {
      display.textContent = this.stateManager.totalClicks;
    }
  }
}

// ============================================
// INITIALIZATION & EVENT HANDLERS
// ============================================

// Initialize global instances using classes from window
const stateManager = new window.ZoneStateManager();
const buttonRenderer = new ZoneButtonRenderer(stateManager, window.ocsCourses);

// Expose state manager globally for compatibility
window.ocsZoneJourney = stateManager;

// Handle increment click
function handleIncrementClick() {
  stateManager.incrementClick();
  buttonRenderer.updateClickDisplay();
  buttonRenderer.updateAllButtons();
}

// Handle clear cookies
function handleClearCookies() {
  stateManager.clear();
  buttonRenderer.updateClickDisplay();
  buttonRenderer.updateAllButtons();
}

// Handle zone selection
function handleZoneSelect(zoneCode) {
  stateManager.selectZone(zoneCode);
  buttonRenderer.updateAllButtons();
}

// Event listeners
document.addEventListener('click', function(e) {
  // Clear cookies button
  if (e.target.id === 'clear-cookies-btn') {
    handleClearCookies();
    return;
  }
  
  // Zone button
  const btn = e.target.closest('.zone-btn');
  if (!btn || btn.disabled) return;
  
  const zone = btn.dataset.zone;
  handleZoneSelect(zone);
});

// Listen for cookie clicks from game
window.addEventListener('cookie-click', handleIncrementClick);

// Initialize display
buttonRenderer.updateClickDisplay();
buttonRenderer.updateAllButtons();
</script>

* **Legacy Navigation:** [Home]({{ site.baseurl }}/home-legacy)

---

{% comment %}
CSSE Zone Section - Start
Hidden container div - shown when CSSE is selected
{% endcomment %}

<div id="zone-section-CSSE" class="zone-section" style="display: none;">
  <script>
  {
    const section = document.getElementById('zone-section-CSSE');
    if (section && window.ocsCourses?.CSSE) {
      const course = window.ocsCourses.CSSE;
      section.innerHTML = `
        <h2>${course.title}</h2>
        <h3 style="margin-top: 0.5rem; color: #667eea;">${course.subtitle}</h3>
        <p style="margin-top: 1rem;">${course.mission}</p>
      `;
    }
  }
  </script>

In [ ]:
%%js

// GAME_RUNNER: CSSE Zone Game | hide_edit: true,  width: 100%, height: 400px

import GameControl from '@assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelCsPath0Forge from '@assets/js/projects/cs-pathway/levels/GameLevelCsPath0Forge.js';
import GameLevelCsPath1Way from '@assets/js/projects/cs-pathway/levels/GameLevelCsPath1Way.js';
import GameLevelCsPath2Mission from '@assets/js/projects/cs-pathway/levels/GameLevelCsPath2Mission.js';
import GameLevelCsPath3Analytics from '@assets/js/projects/cs-pathway/levels/GameLevelCsPath3Analytics.js';


export const gameLevelClasses = [GameLevelCsPath0Forge, GameLevelCsPath1Way, GameLevelCsPath2Mission, GameLevelCsPath3Analytics];
export { GameControl };

{% comment %}
CSSE Zone Section - End
Closes the zone-section-CSSE div
{% endcomment %}

</div><!-- Closes #zone-section-CSSE -->

{% comment %}
CSP Zone Section - Start
Hidden container div - shown when CSP is selected
{% endcomment %}

<div id="zone-section-CSP" class="zone-section" style="display: none;">
  <script>
  {
    const section = document.getElementById('zone-section-CSP');
    if (section && window.ocsCourses?.CSP) {
      const course = window.ocsCourses.CSP;
      section.innerHTML = `
        <h2>${course.title}</h2>
        <h3 style="margin-top: 0.5rem; color: #667eea;">${course.subtitle}</h3>
        <p style="margin-top: 1rem;">${course.mission}</p>
      `;
    }
  }
  </script>

In [ ]:
%%html
<!-- UI_RUNNER: CSP Content | panel: zone_csp, slot: left, layout: row, ratio: 35-65, gap: 1rem -->

<!-- Styles defined in: _sass/open-coding/forms/home-gamified.scss -->
<div class="zone-content" data-zone="CSP">
  <h4>Learning Journey:</h4>
  <ul class="zone-journey-list" id="journey-list-CSP">
    <!-- Populated dynamically from window.ocsCourses -->
  </ul>
</div>

<script>
  const journeyList = document.getElementById('journey-list-CSP');
  if (journeyList) {
    window.ocsCourses.CSP.learningJourney.forEach(item => {
      const li = document.createElement('li');
      li.textContent = item;
      journeyList.appendChild(li);
    });
  }

In [ ]:
%%js

// GAME_RUNNER: CSP Zone Game | hide_edit: true, panel: zone_csp, slot: right, layout: row, ratio: 35-65, gap: 1rem, width: 100%, height: 400px

import GameControl from '@assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelBasic from '@assets/js/GameEnginev1.1/GameLevelBasic.js';


export const gameLevelClasses = [GameLevelBasic];
export { GameControl };

{% comment %}
CSP Zone Section - End
Closes the zone-section-CSP div
{% endcomment %}

</div><!-- Closes #zone-section-CSP -->

{% comment %}
CSA Zone Section - Start
Hidden container div - shown when CSA is selected
{% endcomment %}

<div id="zone-section-CSA" class="zone-section" style="display: none;">
  <script>
  {
    const section = document.getElementById('zone-section-CSA');
    if (section && window.ocsCourses?.CSA) {
      const course = window.ocsCourses.CSA;
      section.innerHTML = `
        <h2>${course.title}</h2>
        <h3 style="margin-top: 0.5rem; color: #667eea;">${course.subtitle}</h3>
        <p style="margin-top: 1rem;">${course.mission}</p>
      `;
    }
  }
  </script>

{% comment %}
CSA Zone Section - End
Closes the zone-section-CSA div
{% endcomment %}

</div><!-- Closes #zone-section-CSA -->

{% comment %}
CSH Zone Section - Start
Hidden container div - shown when CSH is selected
{% endcomment %}

<div id="zone-section-CSH" class="zone-section" style="display: none;">
  <script>
  {
    const section = document.getElementById('zone-section-CSH');
    if (section && window.ocsCourses?.CSH) {
      const course = window.ocsCourses.CSH;
      section.innerHTML = `
        <h2>${course.title}</h2>
        <h3 style="margin-top: 0.5rem; color: #667eea;">${course.subtitle}</h3>
        <p style="margin-top: 1rem;">${course.mission}</p>
      `;
    }
  }
  </script>

In [ ]:
%%html

<!-- UI_RUNNER: CSH Content | panel: zone_csh, slot: left, layout: row, ratio: 35-65, gap: 1rem -->

<!-- Styles defined in: _sass/open-coding/forms/home-gamified.scss -->
<div class="zone-content" data-zone="CSH">
  <h4>Learning Journey:</h4>
  <ul class="zone-journey-list" id="journey-list-CSH">
    <!-- Populated dynamically from window.ocsCourses -->
  </ul>
</div>

<script>
  const journeyList = document.getElementById('journey-list-CSH');
  if (journeyList) {
    window.ocsCourses.CSH.learningJourney.forEach(item => {
      const li = document.createElement('li');
      li.textContent = item;
      journeyList.appendChild(li);
    });
  }
</script>

In [ ]:
%%js

// GAME_RUNNER: CSH Zone Game | hide_edit: true, panel: zone_csh, slot: right, layout: row, ratio: 35-65, gap: 1rem, width: 100%, height: 400px

import GamEnvBackground from '@assets/js/GameEnginev1.1/essentials/GameEnvBackground.js';
import Player from '@assets/js/GameEnginev1.1/essentials/Player.js';
import Npc from '@assets/js/GameEnginev1.1/essentials/Npc.js';
import GameControl from '@assets/js/GameEnginev1.1/essentials/GameControl.js';

class GameLevelCSH {
  constructor(gameEnv) {
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;
    const path = gameEnv.path;

    const bgData = {
      name: 'csh-desert',
      greeting: "Welcome to CSH! Create your capstone project and make an impact!",
      src: path + "/images/gamify/desert.png",
      pixels: { height: 580, width: 1038 }
    };

    const playerData = {
      id: 'CSH Senior',
      greeting: "I'm creating solutions for real-world problems!",
      src: path + "/images/gamify/chillguy.png",
      SCALE_FACTOR: 5,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: 0, y: height - (height/5) },
      pixels: { height: 384, width: 512 },
      orientation: { rows: 3, columns: 4 },
      down: { row: 0, start: 0, columns: 3 },
      left: { row: 2, start: 0, columns: 3 },
      right: { row: 1, start: 0, columns: 3 },
      up: { row: 3, start: 0, columns: 3 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };

    const npcData = {
      id: 'CSH Advisor',
      greeting: "Whirrrr! Ready to solve complex problems and present solutions?",
      src: path + "/images/gamify/r2_idle.png",
      SCALE_FACTOR: 8,
      ANIMATION_RATE: 100,
      pixels: { width: 505, height: 223 },
      INIT_POSITION: { x: width * 1 / 4, y: height * 3 / 4 },
      orientation: { rows: 1, columns: 3 },
      down: { row: 0, start: 0, columns: 3 },
      hitbox: { widthPercentage: 0.1, heightPercentage: 0.2 },
      dialogues: [
        "Your capstone project showcases all your skills!",
        "Real stakeholders need real solutions!",
        "Research deeply before you design!",
        "Documentation ensures your work lives on!",
        "Presenting to experts builds confidence!",
        "Your thesis contributes to the field!"
      ]
    };

    this.classes = [
      { class: GamEnvBackground, data: bgData },
      { class: Player, data: playerData },
      { class: Npc, data: npcData }
    ];
  }
}

export const gameLevelClasses = [GameLevelCSH];
export { GameControl };

{% comment %}
CSH Zone Section - End
Closes the zone-section-CSH div
{% endcomment %}

</div><!-- Closes #zone-section-CSH -->